## [특강] Numpy, Pandas, Matplotlib

### 2. Pandas 기본 구조: Series와 DataFrame

#### 2.1 Series와 DataFrame
Series

- 1차원 구조체 

In [1]:
import pandas as pd

In [3]:
scores = pd.Series([90, 85, 88])

scores, type(scores)

(0    90
 1    85
 2    88
 dtype: int64,
 pandas.Series)

DataFrame

- 2차원(행, 열) 구조체 

In [14]:
data = {
    "이름": ['김철수', '이영희', "박민수"],
    "부서": ["개발", "기획", "개발"],
    "급여": [4500, 5200, 48000]
}

df = pd.DataFrame(data)

df

,이름,부서,급여
0,김철수,개발,4500
1,이영희,기획,5200
2,박민수,개발,48000


#### 2.2 데이터 탐색 속성(shape, dtypes)과 진단(info, describe)

- 탐색적 데이터 분석(EDA)
- df.info()
    - 누락치 확인(Non-Null Count)
    - dtype: 통계가 가능한 자료형인지 체크(잘못된 데이터가 있는지 여부 확인 -> 치환 -> 자료형 변환)
- df.describe()
    - 평균, 중앙값, 최대, 최소값을 통해서 이상치(극단치)를 확인

In [10]:
print("df.info()")
df.info()

df.info()
<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   이름      3 non-null      str  
 1   부서      3 non-null      str  
 2   급여      3 non-null      int64
dtypes: int64(1), str(2)
memory usage: 204.0 bytes


In [15]:
df.describe()

,급여
count,3.000000
mean,19233.333333
std,24915.122583
min,4500.000000
25%,4850.000000
50%,5200.000000
75%,26600.000000
max,48000.000000


#### 2.3 데이터 변경 및 인덱스 제어(set_index, reset_index)

In [18]:
#df['급여'], type(df['급여'])
df['보너스'] = df['급여'] * 0.1

df

,이름,부서,급여,보너스
0,김철수,개발,4500,450.0
1,이영희,기획,5200,520.0
2,박민수,개발,48000,4800.0


In [19]:
# 인덱스를 변경
df_indexed = df.set_index('이름')

df_indexed

,부서,급여,보너스
이름,,,
김철수,개발,4500,450.0
이영희,기획,5200,520.0
박민수,개발,48000,4800.0


In [20]:
# 인덱스를 다시 숫자형 인덱스로 변경
df_restored = df_indexed.reset_index()

df_restored

,이름,부서,급여,보너스
0,김철수,개발,4500,450.0
1,이영희,기획,5200,520.0
2,박민수,개발,48000,4800.0


#### 2.4 Pandas Series & DataFrame 브로드캐스팅 연산

- axis : 0 -> 행의 인덱스 기준
- axis : 1 -> 컬럼 기준

In [24]:
df = pd.DataFrame({
    '중간고사': [80, 95, 70],
    '기말고사': [85, 90, 75]
}, index=['김철수', '이영희', '박민수'])

df

,중간고사,기말고사
김철수,80,85
이영희,95,90
박민수,70,75


In [25]:
score_means = df.mean(axis=0) # 행 기준 (인덱스 기준)
score_means

중간고사    81.666667
기말고사    83.333333
dtype: float64

In [26]:
score_gap = df.sub(score_means, axis=1)

score_gap

,중간고사,기말고사
김철수,-1.666667,1.666667
이영희,13.333333,6.666667
박민수,-11.666667,-8.333333


### 3. Pandas 정밀 조회 및 결측치 제어

#### 3.1  loc와 iloc를 활용한 행·열 선택

- loc : 행의 이름으로 검색(선택)
    - loc[행 라벨]
    - loc[행라벨, 컬럼 이름]
    - loc[시작행:종료행, [컬럼1, 컬럼2, ...]]
    - loc[..., 시작 컬럼:종료컬럼]
- iloc : 숫자 인덱스로 검색(선택)
    - loc와 사용 방법은 동일하지만 행 라벨이 아니라 인덱스 번호
    - 범위를 설정할 경우 종료 번호는 미만
    - iloc[0:2, 0:2] -> 인덱스 0, 1, 컬럼: 0, 1

In [1]:
import pandas as pd
df = pd.DataFrame({
    '나이': [25, 30, 35, 40],
    '점수': [80, 95, 70, 85],
    '등급': ['B', 'A', 'C', 'B']
}, index=['user1', 'user2', 'user3', 'user4'])

df

,나이,점수,등급
user1,25,80,B
user2,30,95,A
user3,35,70,C
user4,40,85,B


In [3]:
df.loc['user1'], type(df.loc['user1'])

(나이    25
 점수    80
 등급     B
 Name: user1, dtype: object,
 pandas.Series)

In [4]:
df.loc['user1', '점수']

np.int64(80)

In [5]:
df.loc['user1':'user3', ['점수', '등급']]

,점수,등급
user1,80,B
user2,95,A
user3,70,C


In [6]:
df.loc['user1':'user2', '점수':'등급']

,점수,등급
user1,80,B
user2,95,A


In [7]:
df.iloc[0:2, 1:3]

,점수,등급
user1,80,B
user2,95,A


#### 3.2 불리언 인덱싱과 query() 조건 검색

불리언 인덱싱

- 컬럼별 값이 True인 조건만 찾아서 조회
- & : and 연산 / 모두 참일때 참
- | : or 연산 / 조건 중 하나만 참이어도 참

In [9]:
df

,나이,점수,등급
user1,25,80,B
user2,30,95,A
user3,35,70,C
user4,40,85,B


In [12]:
(df['나이'] >= 30) & (df['점수'] >= 80)

user1    False
user2     True
user3    False
user4     True
dtype: bool

In [13]:
df[(df['나이'] >= 30) & (df['점수'] >= 80)]

,나이,점수,등급
user2,30,95,A
user4,40,85,B


query() 필터링

In [14]:
df.query('나이 >= 30 and 점수 >= 80')

,나이,점수,등급
user2,30,95,A
user4,40,85,B


#### 3.3 결측치(Missing Data) 7대 처리 기법
3.3.1 결측치 탐지 및 현황 요약 (Detection)

- info()
- isna(), isnull()

In [15]:
import numpy as np

# 결측치 - np.nan, pd.NA, NONE

raw_df = pd.DataFrame({'A': [1, np.nan, 3], 'B': [pd.NA, 5, 6], 'C': [7, 8, 9]})

raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   A       2 non-null      float64
 1   B       2 non-null      object 
 2   C       3 non-null      int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 204.0+ bytes


In [20]:
True + True

2

In [21]:
False + 1

1

In [19]:
raw_df.isna()

,A,B,C
0,False,True,False
1,True,False,False
2,False,False,False


In [22]:
raw_df.isna().sum()

A    1
B    1
C    0
dtype: int64

3.3.2 결측치 삭제 (Drop)

axis - 0:행 기준 - 특정 행에 결측치가 있으면 제거 (기본)
axis - 1:열 기준 - 특정 열에 결측치가 있으면 제거

In [17]:
raw_df

,A,B,C
0,1.0,<NA>,7
1,NaN,5,8
2,3.0,6,9


In [16]:
# 행 기준에서 결측치 삭제
raw_df.dropna()

,A,B,C
2,3.0,6,9


In [18]:
# 열 기준에서 결측치 삭제
raw_df.dropna(axis=1)

,C
0,7
1,8
2,9


3.3.3 통계적 대푯값 대치 (Imputation)

- 평균(mean())
- 중앙값(median()) : 극단치(이상치)의 영향을 덜 받는다.
- fillna(..) 

In [24]:
df_age = pd.DataFrame({'나이': [20, 25, None, 30, 95]})

median_val = df_age['나이'].median()
print("중앙값:", median_val)
df_age['나이'] = df_age['나이'].fillna(median_val)

df_age

중앙값: 27.5


,나이
0,20.0
1,25.0
2,27.5
3,30.0
4,95.0


3.3.4 시계열 직전/직후 값 대치 (Forward / Backward Fill)

- ffill() : 앞쪽에 있는 값으로 대체 
- bfill() : 뒤쪽에 있는 값으로 대체

In [25]:
ts_df = pd.DataFrame({'온도': [18.2, None, None, 21.0]})

# ffill()
ts_df.ffill()

,온도
0,18.2
1,18.2
2,18.2
3,21.0


In [26]:
# bfill()
ts_df.bfill()

,온도
0,18.2
1,21.0
2,21.0
3,21.0


3.3.5 선형 보간법 (Linear Interpolation)

In [27]:
speed_df = pd.DataFrame({'속도': [10.0, None, None, 40.0]})

speed_df['속도'] = speed_df['속도'].interpolate(method='linear')

speed_df

,속도
0,10.0
1,20.0
2,30.0
3,40.0


3.3.6 이상 기호 치환 및 수치형 변환 (Replace & Type Casting)

- replace() : 이상한 데이터를 교체
- pandas.to_numeric(...) : 숫자 자료형으로 변환

In [28]:
survey_df = pd.DataFrame({'점수': [85, -999, '?', 95]})

survey_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   점수      4 non-null      object
dtypes: object(1)
memory usage: 164.0+ bytes


In [29]:
survey_df.describe()

,점수
count,4
unique,4
top,85
freq,1


In [ ]:
survey_df['점수'] = survey_df['점수'].replace([-999, '?'], pd.NA)

survey_df

,점수
0,85
1,<NA>
2,<NA>
3,95


In [32]:
survey_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   점수      2 non-null      object
dtypes: object(1)
memory usage: 164.0+ bytes


In [33]:
survey_df['점수'] = pd.to_numeric(survey_df['점수'], errors="coerce")

survey_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   점수      2 non-null      float64
dtypes: float64(1)
memory usage: 164.0 bytes


In [34]:
survey_df.describe()

,점수
count,2.000000
mean,90.000000
std,7.071068
min,85.000000
25%,87.500000
50%,90.000000
75%,92.500000
max,95.000000


3.3.7 결측 여부 지표 변수 생성 (Missing Indicator)

#### 3.4 그룹화 집계(groupby)와 .to_numpy() 상호 변환